In [ ]:
pip uninstall -y torch torchvision torchaudio

In [ ]:
pip install torch==2.2.2+cu118 torchvision==0.17.2+cu118 --index-url https://download.pytorch.org/whl/cu118

In [ ]:
!pip install numpy==1.26.4 --force-reinstall

In [ ]:
import os
import math
import random
import numpy as np
from glob import glob
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torchvision import transforms
from torchvision.models import vgg16, VGG16_Weights
from tqdm import tqdm

In [ ]:
# -------------------- MODEL: U-Net  --------------------
class DoubleConv(nn.Module):
    """(conv => BN => ReLU) * 2"""
    def __init__(self, in_channels, out_channels, mid_channels=None):
        super().__init__()
        if not mid_channels:
            mid_channels = out_channels
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.double_conv(x)

class Down(nn.Module):
    """Downscaling with maxpool then double conv"""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, out_channels)
        )

    def forward(self, x):
        return self.maxpool_conv(x)

class Up(nn.Module):
    """Upscaling then double conv"""
    def __init__(self, in_channels, out_channels, bilinear=True):
        super().__init__()
        if bilinear:
            self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
            self.conv = DoubleConv(in_channels, out_channels, in_channels // 2)
        else:
            self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
            self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x1, x2):
        x1 = self.up(x1)
        # input is CHW
        diffY = x2.size()[2] - x1.size()[2]
        diffX = x2.size()[3] - x1.size()[3]
        x1 = F.pad(x1, [diffX // 2, diffX - diffX // 2,
                        diffY // 2, diffY - diffY // 2])
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)

class OutConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(OutConv, self).__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=1)

    def forward(self, x):
        return self.conv(x)

class UNet(nn.Module):
    def __init__(self, n_channels=3, n_classes=1, bilinear=True):
        super(UNet, self).__init__()
        self.n_channels = n_channels
        self.n_classes = n_classes
        self.bilinear = bilinear

        self.inc = DoubleConv(n_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        factor = 2 if bilinear else 1
        self.down4 = Down(512, 1024 // factor)
        self.up1 = Up(1024, 512 // factor, bilinear)
        self.up2 = Up(512, 256 // factor, bilinear)
        self.up3 = Up(256, 128 // factor, bilinear)
        self.up4 = Up(128, 64, bilinear)
        self.outc = OutConv(64, n_classes)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)
        logits = self.outc(x)
        return logits


In [ ]:
# -------------------- SEED & TRANSFORMS --------------------
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
set_seed(42)

IMAGE_SIZE = 256
mean = [0.485, 0.456, 0.406]
std  = [0.229, 0.224, 0.225]

transform_img = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

transform_mask_binary = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE), interpolation=Image.NEAREST),
    transforms.PILToTensor()
])

def to_long_binary(mask_tensor):
    if mask_tensor.ndim == 3:
        mask_tensor = mask_tensor.squeeze(0)
    return (mask_tensor > 0).long()

# -------------------- DATASETS --------------------
class SegmentationDatasetA(Dataset):
    """Hard labels (binary masks)"""
    def __init__(self, image_dir, mask_dir, transform_img=None, transform_mask=None):
        self.image_paths = sorted(glob(os.path.join(image_dir, "*.JPG")))
        self.mask_paths = []
        for img_path in self.image_paths:
            basename = os.path.splitext(os.path.basename(img_path))[0]
            mask_path = os.path.join(mask_dir, basename + ".PNG")
            if not os.path.exists(mask_path):
                raise FileNotFoundError(f"Mask not found: {mask_path}")
            self.mask_paths.append(mask_path)
        self.transform_img = transform_img
        self.transform_mask = transform_mask

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert("RGB")
        mask = Image.open(self.mask_paths[idx]).convert("L")
        if self.transform_img:
            img = self.transform_img(img)
        if self.transform_mask:
            mask = self.transform_mask(mask)
        mask = to_long_binary(mask)
        return img, mask, "hard"

class SoftLabelDatasetB(Dataset):
    """Soft labels (teacher probabilities)"""
    def __init__(self, image_dir, soft_label_dir, temperature=1.0, transform_img=None):
        self.image_paths = sorted(glob(os.path.join(image_dir, "*.jpg")))
        self.soft_paths = []
        for img_path in self.image_paths:
            basename = os.path.splitext(os.path.basename(img_path))[0]
            soft_path = os.path.join(soft_label_dir, basename + ".npy")
            if not os.path.exists(soft_path):
                raise FileNotFoundError(f"Soft label not found: {soft_path}")
            self.soft_paths.append(soft_path)
        self.temperature = temperature
        self.transform_img = transform_img

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert("RGB")
        soft_logits = np.load(self.soft_paths[idx])
        if soft_logits.ndim == 3 and soft_logits.shape[-1] in [2, 3, 4]:
            soft_logits = torch.from_numpy(soft_logits).permute(2, 0, 1)
        else:
            soft_logits = torch.from_numpy(soft_logits)
        if soft_logits.shape[1:] != (IMAGE_SIZE, IMAGE_SIZE):
            soft_logits = F.interpolate(soft_logits.unsqueeze(0), size=(IMAGE_SIZE, IMAGE_SIZE),
                                        mode='bilinear', align_corners=False).squeeze(0)
        soft_probs = torch.softmax(soft_logits / self.temperature, dim=0)
        if self.transform_img:
            img = self.transform_img(img)
        return img, soft_probs, "soft"

def collate_mixed(batch):
    imgs = torch.stack([item[0] for item in batch])
    targets = [item[1] for item in batch]
    types = [item[2] for item in batch]
    return imgs, targets, types

class BalancedTrainLoader:
    """Lấy batch gồm cả hard và soft với số lượng cố định mỗi loại"""
    def __init__(self, dataset_hard, dataset_soft, batch_size_hard=4, batch_size_soft=4,
                 shuffle=True, num_workers=0, pin_memory=True, collate_fn=collate_mixed):
        self.loader_hard = DataLoader(dataset_hard, batch_size=batch_size_hard, shuffle=shuffle,
                                      num_workers=num_workers, pin_memory=pin_memory, collate_fn=collate_fn)
        self.loader_soft = DataLoader(dataset_soft, batch_size=batch_size_soft, shuffle=shuffle,
                                      num_workers=num_workers, pin_memory=pin_memory, collate_fn=collate_fn)
        self.batch_size_hard = batch_size_hard
        self.batch_size_soft = batch_size_soft

    def __iter__(self):
        self.iter_hard = iter(self.loader_hard)
        self.iter_soft = iter(self.loader_soft)
        return self

    def __next__(self):
        try:
            batch_hard = next(self.iter_hard)
            batch_soft = next(self.iter_soft)
        except StopIteration:
            raise StopIteration
        imgs_hard, targets_hard, types_hard = batch_hard
        imgs_soft, targets_soft, types_soft = batch_soft
        imgs = torch.cat([imgs_hard, imgs_soft], dim=0)
        targets = targets_hard + targets_soft
        types = list(types_hard) + list(types_soft)
        return imgs, targets, types

    def __len__(self):
        return min(len(self.loader_hard), len(self.loader_soft))

def get_dataloaders(datasetA_root, datasetB_root, temperature=1.0, batch_size=8, num_workers=4, pin_memory=True):
    train_img_dir = os.path.join(datasetA_root, "train/images")
    train_mask_dir = os.path.join(datasetA_root, "train/masks")
    val_img_dir   = os.path.join(datasetA_root, "validation/images")
    val_mask_dir  = os.path.join(datasetA_root, "validation/masks")
    test_img_dir  = os.path.join(datasetA_root, "test/images")
    test_mask_dir = os.path.join(datasetA_root, "test/masks")

    datasetA_train = SegmentationDatasetA(train_img_dir, train_mask_dir, transform_img, transform_mask_binary)
    datasetA_val   = SegmentationDatasetA(val_img_dir, val_mask_dir, transform_img, transform_mask_binary)
    datasetA_test  = SegmentationDatasetA(test_img_dir, test_mask_dir, transform_img, transform_mask_binary)

    datasetB_img_dir = os.path.join(datasetB_root, "images")
    datasetB_soft_dir = os.path.join(datasetB_root, "soft_labels_npy")
    datasetB_train = SoftLabelDatasetB(datasetB_img_dir, datasetB_soft_dir, temperature, transform_img)

    train_loader_hard = DataLoader(datasetA_train, batch_size=batch_size, shuffle=True,
                                   num_workers=num_workers, pin_memory=pin_memory, collate_fn=collate_mixed)
    train_loader_soft = DataLoader(datasetB_train, batch_size=batch_size, shuffle=True,
                                   num_workers=num_workers, pin_memory=pin_memory, collate_fn=collate_mixed)
    val_loader = DataLoader(datasetA_val, batch_size=batch_size, shuffle=False,
                            num_workers=num_workers, pin_memory=pin_memory, collate_fn=collate_mixed)
    test_loader = DataLoader(datasetA_test, batch_size=batch_size, shuffle=False,
                             num_workers=num_workers, pin_memory=pin_memory, collate_fn=collate_mixed)

    print(f"Train hard size: {len(datasetA_train)}, Train soft size: {len(datasetB_train)}")
    print(f"Val size: {len(datasetA_val)}, Test size: {len(datasetA_test)}")
    return train_loader_hard, train_loader_soft, val_loader, test_loader


In [ ]:
# -------------------- LOSS FUNCTIONS --------------------
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, pred, target):
        if pred.dim() == 4 and pred.shape[1] == 1:
            pred = pred.squeeze(1)
        pred_prob = torch.sigmoid(pred)
        target_f = target.float()
        intersection = (pred_prob * target_f).sum(dim=(1,2))
        union = pred_prob.sum(dim=(1,2)) + target_f.sum(dim=(1,2))
        dice = (2.*intersection + self.smooth) / (union + self.smooth)
        return 1. - dice.mean()

class HardLoss(nn.Module):
    def __init__(self, lambda_ce=1.0, lambda_dice=1.0):
        super().__init__()
        self.lambda_ce = lambda_ce
        self.lambda_dice = lambda_dice
        self.dice_fn = DiceLoss()

    def forward(self, logits, targets):
        if logits.dim() == 4 and logits.shape[1] == 1:
            logits = logits.squeeze(1)
        target_tensor = torch.stack(targets, dim=0).to(logits.device)
        if target_tensor.shape[-2:] != logits.shape[-2:]:
            target_tensor = F.interpolate(target_tensor.unsqueeze(1).float(),
                                          size=logits.shape[-2:], mode='nearest').squeeze(1).long()
        l_ce = F.binary_cross_entropy_with_logits(logits, target_tensor.float())
        l_dice = self.dice_fn(logits, target_tensor)
        return self.lambda_ce*l_ce + self.lambda_dice*l_dice, l_ce.item(), l_dice.item()

class KnowledgeDistillationLoss(nn.Module):
    def __init__(self, T=2.0):
        super().__init__()
        self.T = T

    def forward(self, student_logits, teacher_probs):
        device = student_logits.device
        teacher_probs = teacher_probs.to(device)
        if student_logits.shape[1] == 1:
            p_fg = torch.sigmoid(student_logits / self.T)
            p_bg = 1. - p_fg
            student_log_probs = torch.log(torch.cat([p_bg, p_fg], dim=1) + 1e-8)
        else:
            student_log_probs = F.log_softmax(student_logits / self.T, dim=1)

        if teacher_probs.shape[1] != student_log_probs.shape[1]:
            p_fg_t = teacher_probs[:, 1:].sum(dim=1, keepdim=True).clamp(0,1)
            p_bg_t = 1. - p_fg_t
            teacher_probs = torch.cat([p_bg_t, p_fg_t], dim=1)

        if teacher_probs.shape[-2:] != student_log_probs.shape[-2:]:
            teacher_probs = F.interpolate(teacher_probs, size=student_log_probs.shape[-2:],
                                          mode='bilinear', align_corners=False)
        l_kd = -(teacher_probs * student_log_probs).sum(dim=1).mean()
        return (self.T**2) * l_kd

class SoftLoss(nn.Module):
    def __init__(self, T=2.0, lambda_kd=1.0, lambda_dice=1.0):
        super().__init__()
        self.lambda_kd = lambda_kd
        self.lambda_dice = lambda_dice
        self.kd_fn = KnowledgeDistillationLoss(T=T)
        self.dice_fn = DiceLoss()

    def forward(self, logits, soft_probs_list):
        soft_probs = torch.stack(soft_probs_list, dim=0).to(logits.device)
        l_kd = self.kd_fn(logits, soft_probs)
        if soft_probs.shape[1] >= 2:
            pseudo = soft_probs.argmax(dim=1).long()
        else:
            pseudo = (soft_probs.squeeze(1) > 0.5).long()
        if pseudo.shape[-2:] != logits.shape[-2:]:
            pseudo = F.interpolate(pseudo.unsqueeze(1).float(),
                                   size=logits.shape[-2:], mode='nearest').squeeze(1).long()
        logits_sq = logits.squeeze(1) if logits.shape[1]==1 else logits
        l_dice = self.dice_fn(logits_sq, pseudo)
        return self.lambda_kd*l_kd + self.lambda_dice*l_dice, l_kd.item(), l_dice.item()

In [ ]:
# --------------------METRICS --------------------
class SegmentationMetrics:
    def __init__(self, threshold=0.5):
        self.threshold = threshold
        self.reset()

    def reset(self):
        self.tp = self.fp = self.fn = 0

    @torch.no_grad()
    def update(self, logits, targets):
        if logits.dim() == 4 and logits.shape[1] == 1:
            logits = logits.squeeze(1)
        preds = (torch.sigmoid(logits) > self.threshold).long()
        if targets.shape[-2:] != preds.shape[-2:]:
            targets = F.interpolate(targets.unsqueeze(1).float(),
                                    size=preds.shape[-2:], mode='nearest').squeeze(1).long()
        preds = preds.view(-1).cpu()
        targets = targets.view(-1).cpu()
        self.tp += ((preds==1)&(targets==1)).sum().item()
        self.fp += ((preds==1)&(targets==0)).sum().item()
        self.fn += ((preds==0)&(targets==1)).sum().item()

    def compute(self):
        eps = 1e-8
        tp, fp, fn = self.tp, self.fp, self.fn
        return {
            'dice': (2*tp+eps)/(2*tp+fp+fn+eps),
            'iou': (tp+eps)/(tp+fp+fn+eps),
            'precision': (tp+eps)/(tp+fp+eps),
            'recall': (tp+eps)/(tp+fn+eps)
        }


In [ ]:
# -------------------- VALIDATE--------------------
@torch.no_grad()
def validate(model, val_loader, hard_loss_fn, device):
    model.eval()
    metrics = SegmentationMetrics()
    total_loss = 0.0
    n_batches = 0
    for imgs, targets, types in val_loader:
        imgs = imgs.to(device)
        logits = model(imgs)
        if logits.shape[1] > 1:
            logits = logits[:, 1:2]
        logits_up = F.interpolate(logits, size=imgs.shape[-2:], mode='bilinear', align_corners=False)
        hard_idx = [i for i, t in enumerate(types) if t == 'hard']
        if not hard_idx:
            continue
        h_log = logits_up[hard_idx]
        h_tgt = [targets[i] for i in hard_idx]
        loss, _, _ = hard_loss_fn(h_log, h_tgt)
        total_loss += loss.item()
        n_batches += 1
        tgt_stack = torch.stack(h_tgt).to(device)
        metrics.update(h_log, tgt_stack)
    return total_loss / max(n_batches, 1), metrics.compute()


In [ ]:
# -------------------- TRAIN ONE EPOCH --------------------
def train_one_epoch(model, dataloader, loss_fn, is_hard, optimizer, device, epoch, total_epochs):
    model.train()
    loader = dataloader
    total_loss = 0.0
    metrics = SegmentationMetrics()
    n_batches = 0
    if is_hard:
        ce_sum = dice_sum = 0.0
    else:
        kd_sum = dice_sum = 0.0

    expected_type = 'hard' if is_hard else 'soft'
    pbar = tqdm(loader, desc=f"Epoch [{epoch}/{total_epochs}] - {'Hard' if is_hard else 'Soft'}", leave=False)

    for imgs, targets, types in pbar:
        valid_indices = [i for i, t in enumerate(types) if t == expected_type]
        if not valid_indices:
            continue
        imgs = imgs[valid_indices].to(device)
        logits = model(imgs)
        if logits.shape[1] > 1:
            logits = logits[:, 1:2]
        logits_up = F.interpolate(logits, size=imgs.shape[-2:], mode='bilinear', align_corners=False)

        if is_hard:
            tgt = [targets[i] for i in valid_indices]
            loss, l_ce, l_dice = loss_fn(logits_up, tgt)
            ce_sum += l_ce
            dice_sum += l_dice
            tgt_stack = torch.stack(tgt).to(device)
            metrics.update(logits_up, tgt_stack)
            pbar.set_postfix(ce=l_ce, dice=l_dice)
        else:
            soft = [targets[i] for i in valid_indices]
            loss, l_kd, l_dice = loss_fn(logits_up, soft)
            kd_sum += l_kd
            dice_sum += l_dice
            pseudo_labels = []
            for s in soft:
                if s.shape[0] >= 2:
                    pseudo = s.argmax(dim=0).long()
                else:
                    pseudo = (s.squeeze(0) > 0.5).long()
                pseudo_labels.append(pseudo)
            pseudo_stack = torch.stack(pseudo_labels).to(device)
            metrics.update(logits_up, pseudo_stack)
            pbar.set_postfix(kd=l_kd, dice=l_dice)

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
        n_batches += 1

    avg_loss = total_loss / max(n_batches, 1)
    train_metrics = metrics.compute() if n_batches > 0 else {'dice':0, 'iou':0, 'precision':0, 'recall':0}
    if is_hard:
        return {'loss': avg_loss, 'ce': ce_sum/max(n_batches,1), 'dice': dice_sum/max(n_batches,1), 'metrics': train_metrics}
    else:
        return {'loss': avg_loss, 'kd': kd_sum/max(n_batches,1), 'dice': dice_sum/max(n_batches,1), 'metrics': train_metrics}

In [ ]:
# -------------------- FULL TRAINING LOOP --------------------
def train(model, loader_hard, loader_soft, val_loader, device,
          lambda_ce=1.0, lambda_dice=1.0, lambda_kd=1.0, T=2.0,
          lr=6e-5, weight_decay=1e-2, num_epochs=50,
          save_dir="checkpoints", save_best_only=True):

    os.makedirs(save_dir, exist_ok=True)
    hard_loss_fn = HardLoss(lambda_ce=lambda_ce, lambda_dice=lambda_dice)
    soft_loss_fn = SoftLoss(T=T, lambda_kd=lambda_kd, lambda_dice=lambda_dice)

    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=lr*0.01)

    best_dice = 0.0
    history = []

    print(f"\n{'='*68}")
    print(f"  VGG16-AttentionUNet  |  epochs={num_epochs}  |  device={device}")
    print(f"  λ_ce={lambda_ce}  λ_dice={lambda_dice}  λ_kd={lambda_kd}  T={T}")
    print(f"  Alternating hard/soft every epoch")
    print(f"{'='*68}\n")

    for epoch in range(1, num_epochs+1):
        use_hard = (epoch % 2 == 1)
        if use_hard:
            print(f"\n--- Epoch {epoch}/{num_epochs} : Training on HARD dataset ---")
            stats = train_one_epoch(model, loader_hard, hard_loss_fn, True, optimizer, device, epoch, num_epochs)
            print(f"  Train: loss={stats['loss']:.4f}, ce={stats['ce']:.4f}, dice={stats['dice']:.4f}")
            print(f"         dice={stats['metrics']['dice']:.4f}, iou={stats['metrics']['iou']:.4f}, prec={stats['metrics']['precision']:.4f}, rec={stats['metrics']['recall']:.4f}")
        else:
            print(f"\n--- Epoch {epoch}/{num_epochs} : Training on SOFT dataset ---")
            stats = train_one_epoch(model, loader_soft, soft_loss_fn, False, optimizer, device, epoch, num_epochs)
            print(f"  Train: loss={stats['loss']:.4f}, kd={stats['kd']:.4f}, dice={stats['dice']:.4f}")
            print(f"         dice={stats['metrics']['dice']:.4f}, iou={stats['metrics']['iou']:.4f}, prec={stats['metrics']['precision']:.4f}, rec={stats['metrics']['recall']:.4f}")

        val_loss, val_sc = validate(model, val_loader, hard_loss_fn, device)
        scheduler.step()
        lr_now = optimizer.param_groups[0]['lr']

        log = {'epoch': epoch, 'lr': lr_now, 'train_loss': stats['loss'], 'val_loss': val_loss,
               'val_dice': val_sc['dice'], 'val_iou': val_sc['iou'],
               'val_precision': val_sc['precision'], 'val_recall': val_sc['recall']}
        if use_hard:
            log['train_ce'] = stats['ce']; log['train_dice'] = stats['dice']
        else:
            log['train_kd'] = stats['kd']; log['train_dice'] = stats['dice']
        history.append(log)

        print(f"  Val: loss={val_loss:.4f} | dice={val_sc['dice']:.4f}, iou={val_sc['iou']:.4f}, prec={val_sc['precision']:.4f}, rec={val_sc['recall']:.4f}")

        is_best = val_sc['dice'] > best_dice
        if is_best:
            best_dice = val_sc['dice']
            torch.save({
                'epoch': epoch,
                'model_state': model.state_dict(),
                'optim_state': optimizer.state_dict(),
                'val_scores': val_sc
            }, os.path.join(save_dir, "best_model.pth"))
            print(f"  ↳ Best model saved (val_dice={best_dice:.4f})")
        elif not save_best_only:
            torch.save({'epoch': epoch, 'model_state': model.state_dict()},
                       os.path.join(save_dir, f"model_ep{epoch:03d}.pth"))
        print()

    print(f"\nTraining complete. Best val Dice = {best_dice:.4f}")
    return history


In [ ]:
# -------------------- EVALUATE --------------------
@torch.no_grad()
def evaluate(model, test_loader, device):
    model.eval()
    metrics = SegmentationMetrics()
    for imgs, masks, _ in tqdm(test_loader, desc="Testing"):
        imgs = imgs.to(device)
        masks = masks.to(device)
        logits = model(imgs)
        if logits.shape[1] > 1:
            logits = logits[:, 1:2]
        logits_up = F.interpolate(logits, size=imgs.shape[-2:], mode='bilinear', align_corners=False)
        metrics.update(logits_up, masks)
    scores = metrics.compute()
    print("\n===== Test Results =====")
    for k, v in scores.items():
        print(f"  {k:12s}: {v:.4f}")
    return scores


In [ ]:
# -------------------- FULL TRAINING LOOP --------------------
def train(model, loader_hard, loader_soft, val_loader, device,
          lambda_ce=1.0, lambda_dice=1.0, lambda_kd=1.0, T=2.0,
          lr=6e-5, weight_decay=1e-2, num_epochs=50,
          save_dir="checkpoints", save_best_only=True):

    os.makedirs(save_dir, exist_ok=True)
    hard_loss_fn = HardLoss(lambda_ce=lambda_ce, lambda_dice=lambda_dice)
    soft_loss_fn = SoftLoss(T=T, lambda_kd=lambda_kd, lambda_dice=lambda_dice)

    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=lr*0.01)

    best_dice = 0.0
    history = []

    print(f"\n{'='*68}")
    print(f"  VGG16-AttentionUNet  |  epochs={num_epochs}  |  device={device}")
    print(f"  λ_ce={lambda_ce}  λ_dice={lambda_dice}  λ_kd={lambda_kd}  T={T}")
    print(f"  Alternating hard/soft every epoch")
    print(f"{'='*68}\n")

    for epoch in range(1, num_epochs+1):
        use_hard = (epoch % 2 == 1)
        if use_hard:
            print(f"\n--- Epoch {epoch}/{num_epochs} : Training on HARD dataset ---")
            stats = train_one_epoch(model, loader_hard, hard_loss_fn, True, optimizer, device, epoch, num_epochs)
            print(f"  Train: loss={stats['loss']:.4f}, ce={stats['ce']:.4f}, dice={stats['dice']:.4f}")
            print(f"         dice={stats['metrics']['dice']:.4f}, iou={stats['metrics']['iou']:.4f}, prec={stats['metrics']['precision']:.4f}, rec={stats['metrics']['recall']:.4f}")
        else:
            print(f"\n--- Epoch {epoch}/{num_epochs} : Training on SOFT dataset ---")
            stats = train_one_epoch(model, loader_soft, soft_loss_fn, False, optimizer, device, epoch, num_epochs)
            print(f"  Train: loss={stats['loss']:.4f}, kd={stats['kd']:.4f}, dice={stats['dice']:.4f}")
            print(f"         dice={stats['metrics']['dice']:.4f}, iou={stats['metrics']['iou']:.4f}, prec={stats['metrics']['precision']:.4f}, rec={stats['metrics']['recall']:.4f}")

        val_loss, val_sc = validate(model, val_loader, hard_loss_fn, device)
        scheduler.step()
        lr_now = optimizer.param_groups[0]['lr']

        log = {'epoch': epoch, 'lr': lr_now, 'train_loss': stats['loss'], 'val_loss': val_loss,
               'val_dice': val_sc['dice'], 'val_iou': val_sc['iou'],
               'val_precision': val_sc['precision'], 'val_recall': val_sc['recall']}
        if use_hard:
            log['train_ce'] = stats['ce']; log['train_dice'] = stats['dice']
        else:
            log['train_kd'] = stats['kd']; log['train_dice'] = stats['dice']
        history.append(log)

        print(f"  Val: loss={val_loss:.4f} | dice={val_sc['dice']:.4f}, iou={val_sc['iou']:.4f}, prec={val_sc['precision']:.4f}, rec={val_sc['recall']:.4f}")

        is_best = val_sc['dice'] > best_dice
        if is_best:
            best_dice = val_sc['dice']
            torch.save({
                'epoch': epoch,
                'model_state': model.state_dict(),
                'optim_state': optimizer.state_dict(),
                'val_scores': val_sc
            }, os.path.join(save_dir, "best_model.pth"))
            print(f"  ↳ Best model saved (val_dice={best_dice:.4f})")
        elif not save_best_only:
            torch.save({'epoch': epoch, 'model_state': model.state_dict()},
                       os.path.join(save_dir, f"model_ep{epoch:03d}.pth"))
        print()

    print(f"\nTraining complete. Best val Dice = {best_dice:.4f}")
    return history


In [ ]:
# -------------------- EVALUATE --------------------
@torch.no_grad()
def evaluate(model, test_loader, device):
    model.eval()
    metrics = SegmentationMetrics()
    for imgs, masks, _ in tqdm(test_loader, desc="Testing"):
        imgs = imgs.to(device)
        masks = masks.to(device)
        logits = model(imgs)
        if logits.shape[1] > 1:
            logits = logits[:, 1:2]
        logits_up = F.interpolate(logits, size=imgs.shape[-2:], mode='bilinear', align_corners=False)
        metrics.update(logits_up, masks)
    scores = metrics.compute()
    print("\n===== Test Results =====")
    for k, v in scores.items():
        print(f"  {k:12s}: {v:.4f}")
    return scores


In [ ]:
# -------------------- MAIN --------------------
if __name__ == "__main__":
    datasetA_root = "/kaggle/input/datasets/totrang24082004/otu2d-dataset/dataset_split"
    datasetB_root = "/kaggle/input/datasets/totrang24082004/sample"

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # ---------- SỬ DỤNG U-Net (thay vì VGG16AttentionUNet) ----------
    model = UNet(n_channels=3, n_classes=1, bilinear=True).to(device)

    train_loader_hard, train_loader_soft, val_loader, test_loader = get_dataloaders(
        datasetA_root, datasetB_root,
        temperature=2.0,
        batch_size=10,
        num_workers=2,
        pin_memory=True
    )

    history = train(
        model=model,
        loader_hard=train_loader_hard,
        loader_soft=train_loader_soft,
        val_loader=val_loader,
        device=device,
        lambda_ce=1.0,
        lambda_dice=0.5,
        lambda_kd=0.1,
        T=2.0,
        lr=1e-4,
        weight_decay=1e-2,
        num_epochs=50,
        save_dir="checkpoints",
        save_best_only=True
    )

    # Load best và đánh giá test
    model.load_state_dict(torch.load("checkpoints/best_model.pth")['model_state'])
    test_scores = evaluate(model, test_loader, device)
